# 1 2 pre correction nmr count

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()


In [ ]:
import pandas as pd
# The supplied event-level data already contain prior-NMR counts.
source_corrections = pd.read_parquet('data/source_post_features_processed.parquet')
horizons = [4, 28, 60, 90, 180]
nmr_count_columns = [f'nmr_count_{days}d' for days in horizons]
print(source_corrections.shape)
display(source_corrections[nmr_count_columns].describe())


In [ ]:
source_corrections = source_corrections.query('prior_corrections==0').reset_index(drop=True)
print(source_corrections.shape)

In [ ]:
display(source_corrections[nmr_count_columns].describe())

### Prior NMR notes before the first correction
Each horizon uses its own eligible accounts: `days_from_rollout > horizon_days` for 4, 28, 60, 90, and 180 days. Both plots use that horizon-specific denominator; sample sizes may differ across horizons. Ineligible counts are missing, not zero. The current high-score cutoff in the counting cell is 0.30.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import FONT_SIZE
from matplotlib.ticker import PercentFormatter, MultipleLocator

plot_accounts = source_corrections.query('treated == 1 and current_correction == 1').copy()
assert plot_accounts['user_id'].is_unique, 'Expected one first correction per account.'
assert len(plot_accounts) > 0, 'No eligible first-corrected accounts.'
horizons = [4, 28, 60, 90, 180]

def plot_prior_nmr(count_prefix, minimum, ylabel, filename):
    rows = []
    for days in horizons:
        eligible_accounts = plot_accounts.loc[plot_accounts[f'nmr_eligible_{days}d']]
        rows.append({
            'horizon_days': days,
            'n_accounts': len(eligible_accounts),
            'n_qualifying_accounts': eligible_accounts[f'{count_prefix}_{days}d'].ge(minimum).sum(),
        })
    summary = pd.DataFrame(rows)
    summary['percent_accounts'] = 100 * summary['n_qualifying_accounts'] / summary['n_accounts'].replace(0, np.nan)
    display(summary.round(2))
    plt.style.use('default')
    plt.rcParams['font.size'] = FONT_SIZE
    fig, ax = plt.subplots(figsize=(8.5, 5), layout='constrained')
    positions = np.arange(len(horizons))
    bars = ax.bar(positions, summary['percent_accounts'], width=0.34,
                  color='grey', edgecolor='black', linewidth=1, hatch='//', zorder=3)
    ax.bar_label(bars, labels=[f'{bar.get_height():.1f}%' for bar in bars],
                 padding=5, fontsize=FONT_SIZE-3, color='black')
    ax.set_xticks(positions)
    ax.set_xticklabels(horizons)
    ax.set_xlabel('Classification horizon (days)')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_locator(MultipleLocator(20))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.grid(axis='y', color='lightgrey', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', length=0, pad=8)
    fig.savefig(f'outputs/figures/{filename}.pdf')
    plt.show()
    return summary


nmr_two_summary = plot_prior_nmr(
    'nmr_count', 2, 'NMR-note recurrence ($\geq 2$)\n(% of eligible accounts)',
    'pre_correction_nmr_at_least_two',
)
nmr_high_score_summary = plot_prior_nmr(
    'nmr_high_score_count', 1,
    'High-score NMR notes ($\geq 1$)\n(% of eligible accounts)',
    'pre_correction_nmr_high_score_at_least_one',
)